In [31]:
import numpy as np

In [32]:
def eliminacao_gauss_aumentada(dim,A,b):
  #M é matriz aumentada [A|b]
  M = np.hstack((A, b.reshape(-1, 1)))

  for k in range(dim-1):
    for i in range(k+1, dim):
      m = M[i][k] / M[k][k]
      for j in range(k, dim + 1):  # +1 para incluir a coluna b
        M[i][j] = M[i][j] - m * M[k][j]

  # Substituição retroativa
  x = np.zeros(dim)
  for i in range(dim - 1, -1, -1):
    #A última coluna de M é o vetor b modificado
    x[i] = (M[i,dim] - np.sum(M[i,i+1:dim]*x[i+1:dim]))/M[i][i]
  return x

#Exemplo
A = np.array([[6, 2, -1], [2, 4, 1], [3, 2, 8]])
b = np.array([7, 7, 13])


x_solucao = eliminacao_gauss_aumentada(3,A,b)
print("\nSolução x:")
print(x_solucao)


Solução x:
[1. 1. 1.]


In [33]:
def eliminacao_gauss(A,b):
  dim = len(b)

  for k in range (dim-1):
    for i in range (k+1,dim):
      m =  A[i][k]/A[k][k]
      for j in range (k,dim):
        A[i][j] = A[i][j] - m* A[k][j]
      b[i] = b[i] - m*b[k]

  #Voltando - substituição retroativa
  x = np.zeros(dim)
  #Aqui o dim-1 mostra q ele vai de tras para frente
  #O 1° -1 é que ele vai fazer o for até o -1, fazendo o ultimo loop com i=0
  #O 2° -1 indica que esta decrementando
  for i in range (dim-1,-1,-1):
    soma = 0.0
    for j in range (i+1,dim):
      soma += A[i,j] * x[j]
    #Cálculo de x[i]
    x[i] = (b[i] - soma)/A[i,i]

  return x

#Exemplo
A = np.array([[6,2,-1], [2,4,1], [3,2,8]])
b = np.array([7,7,13])
x = eliminacao_gauss(A,b)
print(x)

[1. 1. 1.]


In [34]:
def decomposicao_LU(A,b):
  L = np.zeros(A.shape)
  U = np.zeros(A.shape)
  lengh = len(b)
  for i in range(lengh):
    L[i,i] = 1.0

  for i in range(lengh):
    #Cálculo de U (tringular superior)
    for j in range(i,lengh):
      sum = 0.0
      for k in range(i):
        sum += L[i,k]*U[k,j]
      U[i,j] = A[i,j] - sum

    #Cálculo de L (tringular inferior)
    for j in range (i+1,lengh):
      sum_2 = 0.0
      for k in range(i):
        sum_2 += L[j,k]*U[k,i]
      L[j,i] = (A[j,i] - sum_2)/U[i,i]

  #Resolvendo Ly=b
  y = np.zeros(lengh)
  for i in range(lengh):
    sum_3 = 0.0
    for k in range(i):
      sum_3 += L[i,k]*y[k]
    y[i] = (b[i] - sum_3)/L[i][i]

  #Resolvendo Ux = y por substituição retroativa
  x = np.zeros(lengh)
  for i in range(lengh-1,-1,-1):
    sum_4 = 0.0
    for k in range(i+1,lengh):
      sum_4 += U[i,k]*x[k]
    x[i] = (y[i] - sum_4)/U[i,i]
  return x

A = np.array([[5,2,1], [3,1,4], [1,1,3]], dtype=float)
b = np.array([0,-7,-5], dtype=float)
x = decomposicao_LU(A,b)
print(np.round(x, decimals=10))

[-0.  1. -2.]


In [35]:
def decomposicao_Cholesky(A,b):
  lengh = len(b)
  G = np.zeros((lengh,lengh))

  #Fazendo A = G*G^t
  for i in range (lengh):
    for j in range(i+1):
      soma=0.0
      if(i==j):
        for k in range(j):
          soma += G[i,k]**2
        G[i,j] = np.sqrt(A[i,i] - soma)
      else:
        for k in range(j):
          soma += G[i,k]*G[j,k]
        G[i,j]= (A[i,j] - soma)/G[j,j]

  #Resolvendo Gy=b
  y = np.zeros(lengh)

  for i in range (lengh):
    som_2 = 0.0
    for k in range (i):
      som_2 += G[i,k]*y[k]
    y[i] = (b[i] - som_2)/G[i][i]

  x = np.zeros(lengh)
  for i in range (lengh-1,-1,-1):
    som_3 = 0.0
    for k in range(i+1,lengh):
      som_3 += G[k,i]*x[k]
    x[i] = (y[i] - som_3)/G[i,i]
  return x

A = np.array([[4.,2,-4], [2,10,4], [-4,4,9]])
b = np.array([0.,6.,5.])
x = decomposicao_Cholesky(A,b)
print(x)

[1. 0. 1.]


In [36]:
def verifca_LU(A,b):
    lengh = len(b)
    #Verficar se A é quadrada
    if A.shape[0] != A.shape[1]:
        print("A matriz A deve ser quadrada.")
        return False
    #verificar se os determinantes dos menores principais são diferentes de zero
    for i in range(1,lengh):
        #Python pega do índice 0 até o índice i-1 no slicing, pegando a matriz de ordem i
        if np.linalg.det(A[:i,:i]) == 0:
            print(f"O determinante do menor principal de ordem {i} é zero. A decomposição LU não é possível.")
            return False
    return True

In [37]:
def verifica_Cholesky(A,b):
    lengh = len(b)
    #Verificar se A é simétrica
    A_transposta = A.T
    for i in range(lengh):
        for j in range(lengh):
            if A[i][j] != A_transposta[i][j]:
                print("A matriz A deve ser simétrica.")
                print(f"A[{i}][{j}] = {A[i][j]} != A^T[{i}][{j}] = {A_transposta[i][j]}")
                return False
    # if not np.allclose(A, A_transposta):
    #     print("A matriz A deve ser simétrica.")
    #     return False
    #Verificar se A é positiva definida
    # if not np.all(np.linalg.eigvals(A) > 0):
    #     print("A matriz A deve ser positiva definida.")
    #     return False
    for i in range(1,lengh):
    #Python pega do índice 0 até o índice i-1 no slicing, pegando a matriz de ordem i
        if np.linalg.det(A[:i,:i]) <= 0:
            print(f"O determinante do menor principal de ordem {i} é zero ou negativo. A decomposição LU não é possível.")
            return False
    return True

In [38]:
#Exercício
Matriz_A = np.array([[4.,12.,-16], [12,37,-43.], [-16.,-43,98]])
Matriz_b = np.array([1.,2,3])

#A) Verifique se este sistema satisfaz as condições para utilização dos métodos de Decomposição LU e Cholesky, explicando sua resposta.
print("Verificando condições para Decomposição LU:")
if verifca_LU(Matriz_A,Matriz_b):
    print("O sistema satisfaz as condições para a Decomposição LU.")
    print("\nA função verifica se a matriz A fornecidada é quadrada, ela verifica se o número re linhas e colunas são iguais. ")
    print("Em seguida, ela verifica se os determinantes dos menores principais de A são diferentes de zero, o que é necessário\npara garantir que a decomposição LU seja possível.\n")
    print("Se todas as condições forem satisfeitas, a função retorna True, indicando que a decomposição LU pode ser aplicada ao sistema.")
else:
    print("O sistema não satisfaz as condições para a Decomposição LU.")

print("\n\nVerificando condições para Decomposição Cholesky:")
if verifica_Cholesky(Matriz_A,Matriz_b):
    print("O sistema satisfaz as condições para a Decomposição Cholesky.")
    print("\nA função verifica se a matriz A fornecida é simétrica, comparando cada elemento de A com seu correspondente na transposta de A. ")
    print("Se encontrar algum elemento que não seja igual ao seu correspondente na transposta, a função imprime uma mensagem indicando que\na matriz não é simétrica e retorna False.\n")
    print("Em seguida, a função verifica se a matriz A é positiva definida, calculando se o determinante dos menores principais de A são \nmaiores que zero. ")
    print("Se encontrar algum determinante que não seja maior que zero, a função imprime uma mensagem indicando que a matriz não é positiva\ndefinida e retorna False.\n")
    print("Se ambas as condições forem satisfeitas, a função retorna True, indicando que a decomposição Cholesky pode ser aplicada ao sistema.")
else:
    print("O sistema não satisfaz as condições para a Decomposição Cholesky.")




Verificando condições para Decomposição LU:
O sistema satisfaz as condições para a Decomposição LU.

A função verifica se a matriz A fornecidada é quadrada, ela verifica se o número re linhas e colunas são iguais. 
Em seguida, ela verifica se os determinantes dos menores principais de A são diferentes de zero, o que é necessário
para garantir que a decomposição LU seja possível.

Se todas as condições forem satisfeitas, a função retorna True, indicando que a decomposição LU pode ser aplicada ao sistema.


Verificando condições para Decomposição Cholesky:
O sistema satisfaz as condições para a Decomposição Cholesky.

A função verifica se a matriz A fornecida é simétrica, comparando cada elemento de A com seu correspondente na transposta de A. 
Se encontrar algum elemento que não seja igual ao seu correspondente na transposta, a função imprime uma mensagem indicando que
a matriz não é simétrica e retorna False.

Em seguida, a função verifica se a matriz A é positiva definida, calculando 

In [39]:
#B)Utilizando os scripts desenvolvidos, obtenha solução numérica para esse sistema de equações lineares pelos métodos de
#Eliminação de Gauss e pelos métodos de Decomposição LU e Cholesky se forem viáveis.

print("\n\nSolução utilizando Eliminação de Gauss:")
solucao_gauss = eliminacao_gauss(Matriz_A.copy(),Matriz_b.copy())
print(f"Solução de solucao_gauss: {solucao_gauss}")

print("\nSolução utilizando Decomposição LU:")
if verifca_LU(Matriz_A,Matriz_b):
    solucao_LU = decomposicao_LU(Matriz_A.copy(),Matriz_b.copy())
    print(f"Solução de solucao_LU: {solucao_LU}")
else:
    print("A Decomposição LU não é viável para este sistema.")

print("\nSolução utilizando Decomposição Cholesky:")
if verifica_Cholesky(Matriz_A,Matriz_b):
    solucao_Cholesky = decomposicao_Cholesky(Matriz_A.copy(),Matriz_b.copy())
    print(f"Solução de solucao_Cholesky: {solucao_Cholesky}")
else:
    print("A Decomposição Cholesky não é viável para este sistema.")



Solução utilizando Eliminação de Gauss:
Solução de solucao_gauss: [28.58333333 -7.66666667  1.33333333]

Solução utilizando Decomposição LU:
Solução de solucao_LU: [28.58333333 -7.66666667  1.33333333]

Solução utilizando Decomposição Cholesky:
Solução de solucao_Cholesky: [28.58333333 -7.66666667  1.33333333]
